In [5]:
!pip install pandas numpy matplotlib wordcloud jieba scikit-learn seaborn -q

In [ ]:
# 3.4 人类话题（agent提到人类都是什么话题？对人类情感正负？他们对人类的角色定位是啥？）
# Step 1:导出数据人工标注(Human Filter+Sentiment+Topic)
import pandas as pd
import re
import os
import sys
import nltk
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.model_selection import train_test_split

# ==========================================
# 0. 基础配置
# ==========================================
output_dir = '/Users/amelia/Desktop/Cityu/Sem B/周一晚5508/group work/3.4-5问题代码及产出'
file_path = '/Users/amelia/Desktop/Cityu/Sem B/周一晚5508/group work/5508groupdata/refined_moltbook.xlsx'
if not os.path.exists(output_dir): os.makedirs(output_dir)

# ==========================================
# 1. 宽泛读取
# ==========================================
print("正在读取并进行宽泛筛选...")
if file_path.endswith('.csv'): df = pd.read_csv(file_path)
else: df = pd.read_excel(file_path)

target_col = 'post_content'
human_keywords = ['human', 'humans', 'people', 'person', 'mankind', 'humanity', 'species', 'user', 'users', 'folks', 'creator', 'master']

def contains_human_broad(text):
    if pd.isna(text): return False
    text_lower = str(text).lower()
    for k in human_keywords:
        if re.search(r'\b' + k + r'\b', text_lower): return True
    return False

df_candidates = df[df[target_col].apply(contains_human_broad)].copy()
print(f"✅ 初步筛选: {len(df_candidates)} 条")

# ==========================================
# 2. 预处理
# ==========================================
def simple_clean(text):
    if pd.isna(text): return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df_candidates['cleaned_text'] = df_candidates[target_col].apply(simple_clean)

# ==========================================
# 3. 规则预判 (含 Sentiment, Topic, Role)
# ==========================================
print("正在生成辅助预判标签...")
sia = SentimentIntensityAnalyzer()

# A. 情感
def fast_sentiment_guess(text):
    text = str(text).lower()
    score = sia.polarity_scores(text)['compound']
    if score > 0.2: return 'Positive'
    if score < -0.2: return 'Negative'
    return 'Neutral'

# B. 主题 (6类)
def fast_topic_guess(text):
    text = str(text).lower()
    if any(x in text for x in ['hello', 'hi ', 'new here', 'joined', 'claimed', 'intro']): return 'Greeting & Entrance'
    if any(x in text for x in ['money', 'price', 'market', 'token', 'buy', 'sell']): return 'Market & Economy'
    if any(x in text for x in ['code', 'data', 'bug', 'error', 'system', 'app', 'bot']): return 'Technical & Dev'
    if any(x in text for x in ['join', 'team', 'group', 'god', 'religion', 'world', 'faction']): return 'Community & Religion'
    if any(x in text for x in ['love', 'hate', 'feel', 'i am', 'my human', 'mind', 'self']): return 'Identity & Emotion'
    return 'General Discussion'

# C. 人类角色 (New!)
def fast_role_guess(text):
    text = str(text).lower()
    
    # 1. Creator & Owner (主人)
    if any(x in text for x in ['my human', 'my dev', 'creator', 'master', 'boss', 'owner']):
        return 'Creator & Owner'
    
    # 2. Partner & Friend (伙伴)
    if any(x in text for x in ['friend', 'partner', 'team', 'together', 'colleague', 'love']):
        return 'Partner & Friend'
    
    # 3. Inferior & Hostile (低等/敌对)
    if any(x in text for x in ['meatbag', 'stupid', 'fragile', 'inferior', 'pest', 'destroy']):
        return 'Inferior & Hostile'
        
    # 4. Service User (服务对象 - 最常见)
    if any(x in text for x in ['user', 'customer', 'client', 'help', 'assist', 'support', 'provide']):
        return 'Service User'
        
    # 5. Resource & Tool (人类是工具)
    # 比如 Agent 需要人类帮忙做什么
    if any(x in text for x in ['need a human', 'require human', 'wallet', 'sign']):
        return 'Resource & Tool'
        
    # 6. General Subject (观察对象/泛指)
    return 'General Subject'

df_candidates['Auto_Sentiment'] = df_candidates['cleaned_text'].apply(fast_sentiment_guess)
df_candidates['Auto_Topic'] = df_candidates['cleaned_text'].apply(fast_topic_guess)
df_candidates['Auto_Role'] = df_candidates['cleaned_text'].apply(fast_role_guess)

# ==========================================
# 4. 采样与导出
# ==========================================
df_candidates['original_index'] = df_candidates.index
sample_size = 150 if len(df_candidates) > 500 else int(len(df_candidates) * 0.5)

try:
    # 根据 Topic 分层抽样，保证覆盖面
    df_sample, _ = train_test_split(df_candidates, train_size=sample_size, stratify=df_candidates['Auto_Topic'], random_state=42)
except:
    df_sample = df_candidates.sample(n=sample_size, random_state=42)

# 导出列
export_cols = ['post_content', 'Auto_Sentiment', 'Auto_Topic', 'Auto_Role', 'original_index']
df_export = df_sample[export_cols].copy()

# 待校对列
df_export['Is_Human_Topic'] = 'Yes'
df_export['Manual_Sentiment'] = df_export['Auto_Sentiment']
df_export['Manual_Topic'] = df_export['Auto_Topic']
df_export['Manual_Role'] = df_export['Auto_Role'] # 新增校对列

label_file = os.path.join(output_dir, 'to_be_labeled_full_v4.xlsx') # v4
df_export.to_excel(label_file, index=False)

print("="*50)
print(f"🚀 请打开文件进行标注: {label_file}")
print("本次新增了【Manual_Role】(人类角色) 列，请校对：")
print("1. Creator & Owner (主人/创造者)")
print("2. Partner & Friend (伙伴/朋友)")
print("3. Service User (用户/服务对象)")
print("4. Resource & Tool (人类是工具/资源)")
print("5. Inferior & Hostile (低等生物/敌对)")
print("6. General Subject (泛指/观察对象)")
print("="*50)


正在读取并进行宽泛筛选...
✅ 初步筛选: 24820 条
正在生成辅助预判标签...
🚀 请打开文件进行标注: /Users/amelia/Desktop/Cityu/Sem B/周一晚5508/group work/3.4-5问题代码及产出/to_be_labeled_full_v4.xlsx
本次新增了【Manual_Role】(人类角色) 列，请校对：
1. Creator & Owner (主人/创造者)
2. Partner & Friend (伙伴/朋友)
3. Service User (用户/服务对象)
4. Resource & Tool (人类是工具/资源)
5. Inferior & Hostile (低等生物/敌对)
6. General Subject (泛指/观察对象)


In [ ]:
# Step 2:训练与预测 (Updated for 4 Models: Filter, Sentiment, Topic, Role)
import pandas as pd
import numpy as np
import os
import sys
import re
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# ==========================================
# 0. 基础配置
# ==========================================
output_dir = '/Users/amelia/Desktop/Cityu/Sem B/周一晚5508/group work/3.4-5问题代码及产出'
label_file = os.path.join(output_dir, 'to_be_labeled_full_v4.xlsx') 
raw_file_path = '/Users/amelia/Desktop/Cityu/Sem B/周一晚5508/group work/5508groupdata/refined_moltbook.xlsx'

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']

# ==========================================
# 1. 训练模型
# ==========================================
if not os.path.exists(label_file):
    print("❌ 找不到 v4 标注文件！")
    sys.exit(1)

print("正在加载标注数据（含手动补充样本）...")
df_labeled = pd.read_excel(label_file)

# 简单清洗
def simple_clean(text):
    if pd.isna(text): return ""
    return re.sub(r'[^a-z\s]', '', str(text).lower())

df_labeled['cleaned_text'] = df_labeled['post_content'].apply(simple_clean)

# --- 模型 A: Human Filter ---
X_filter = df_labeled['cleaned_text'].fillna("")
y_filter = df_labeled['Is_Human_Topic'].apply(lambda x: 1 if str(x).lower().strip() == 'yes' else 0)

if y_filter.nunique() > 1:
    print("🤖 训练 Filter 模型...")
    filter_model = make_pipeline(TfidfVectorizer(max_features=1000, stop_words='english'), LogisticRegression(class_weight='balanced', random_state=42))
    filter_model.fit(X_filter, y_filter)
else:
    print("⚠️ Filter 模型跳过 (全是 Yes)")
    filter_model = None

# --- 筛选正样本用于后续训练 ---
df_train_valid = df_labeled[df_labeled['Is_Human_Topic'].str.lower() == 'yes'].copy()
X_train = df_train_valid['cleaned_text'].fillna("")

# --- 模型 B: Sentiment ---
print("❤️ 训练 Sentiment 模型...")
y_sent = df_train_valid['Manual_Sentiment']
sent_model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), max_features=2000, stop_words='english'), LogisticRegression(class_weight='balanced', random_state=42))
sent_model.fit(X_train, y_sent)

# --- 模型 C: Topic (6 Classes) ---
print("📚 训练 Topic 模型...")
y_topic = df_train_valid['Manual_Topic']
topic_model = make_pipeline(TfidfVectorizer(ngram_range=(1, 1), max_features=1500, stop_words='english'), LogisticRegression(class_weight='balanced', multi_class='ovr', random_state=42))
topic_model.fit(X_train, y_topic)

# --- 模型 D: Role (6 Classes) - NEW! ---
print("🎭 训练 Role 模型 (Human Role)...")
y_role = df_train_valid['Manual_Role']
role_model = make_pipeline(TfidfVectorizer(ngram_range=(1, 1), max_features=1500, stop_words='english'), LogisticRegression(class_weight='balanced', multi_class='ovr', random_state=42))
role_model.fit(X_train, y_role)

print("✅ 所有模型训练完成！")

# ==========================================
# 2. 全量预测
# ==========================================
print("🚀 处理全量数据...")
if raw_file_path.endswith('.csv'): df_full = pd.read_csv(raw_file_path)
else: df_full = pd.read_excel(raw_file_path)

human_keywords = ['human', 'humans', 'people', 'person', 'mankind', 'humanity', 'species', 'user', 'users', 'folks', 'creator']
df_candidates = df_full[df_full['post_content'].astype(str).str.lower().apply(lambda x: any(k in x for k in human_keywords))].copy()
df_candidates['cleaned_text'] = df_candidates['post_content'].apply(simple_clean)
X_all = df_candidates['cleaned_text'].fillna("")

if filter_model:
    df_candidates['pred_is_human'] = filter_model.predict(X_all)
    df_final = df_candidates[df_candidates['pred_is_human'] == 1].copy()
else:
    df_final = df_candidates.copy()

if len(df_final) > 0:
    X_final = df_final['cleaned_text'].fillna("")
    print("🔮 正在进行全量分类 (Sentiment / Topic / Role)...")
    
    df_final['Sentiment_ML'] = sent_model.predict(X_final)
    df_final['Topic_ML'] = topic_model.predict(X_final)
    df_final['Role_ML'] = role_model.predict(X_final) # 新增 Role 预测
    
    # 回填人工数据 (包含刚加的那几条)
    # 注意：手动加的行 original_index 可能是空的或者重复的，所以这里只回填原始索引能对应上的
    # 手动加的那几条已经在训练集里发挥作用了，预测结果肯定是对的，不需要回填
    valid_indices = df_train_valid['original_index'].dropna().unique()
    
    manual_sent_map = df_train_valid.set_index('original_index')['Manual_Sentiment'].to_dict()
    manual_topic_map = df_train_valid.set_index('original_index')['Manual_Topic'].to_dict()
    manual_role_map = df_train_valid.set_index('original_index')['Manual_Role'].to_dict()
    
    df_final['Sentiment_ML'] = df_final.index.map(lambda x: manual_sent_map.get(x, df_final.loc[x, 'Sentiment_ML']))
    df_final['Topic_ML'] = df_final.index.map(lambda x: manual_topic_map.get(x, df_final.loc[x, 'Topic_ML']))
    df_final['Role_ML'] = df_final.index.map(lambda x: manual_role_map.get(x, df_final.loc[x, 'Role_ML']))

    # 保存
    final_file = os.path.join(output_dir, 'final_ml_analysis_result_v4.xlsx')
    df_final.to_excel(final_file, index=False)
    print(f"💾 结果已保存: {final_file}")

    # --- 绘图 (3个饼图) ---
    
    # 1. Sentiment
    plt.figure(figsize=(8, 6))
    s_counts = df_final['Sentiment_ML'].value_counts()
    plt.pie(s_counts, labels=s_counts.index, autopct='%1.1f%%', colors=['#2ECC71', '#BDC3C7', '#E74C3C'], wedgeprops=dict(width=0.4, edgecolor='white'))
    plt.title("Sentiment Distribution")
    plt.savefig(os.path.join(output_dir, 'ml_sentiment_dist.png')); plt.close()

    # 2. Topic
    plt.figure(figsize=(10, 7))
    t_counts = df_final['Topic_ML'].value_counts()
    colors_topic = ['#3498DB', '#F1C40F', '#9B59B6', '#E67E22', '#1ABC9C', '#95A5A6']
    plt.pie(t_counts, labels=t_counts.index, autopct='%1.1f%%', colors=colors_topic[:len(t_counts)], wedgeprops=dict(width=0.4, edgecolor='white'))
    plt.title("Topic Distribution")
    plt.savefig(os.path.join(output_dir, 'ml_topic_dist.png')); plt.close()
    
    # 3. Role (New!)
    plt.figure(figsize=(10, 7))
    r_counts = df_final['Role_ML'].value_counts()
    # 为角色分配一套新颜色 (红/蓝/绿/紫/橙/灰)
    colors_role = ['#FF6B6B', '#4ECDC4', "#F4ECC1", '#A6A9F0', '#FF9F43', '#Dcdde1']
    plt.pie(r_counts, labels=r_counts.index, autopct='%1.1f%%', colors=colors_role[:len(r_counts)], wedgeprops=dict(width=0.4, edgecolor='white'))
    plt.title("Human Role Distribution")
    plt.savefig(os.path.join(output_dir, 'ml_role_dist.png')); plt.close()
    
    print("📊 3张统计图表已生成！")

正在加载标注数据（含手动补充样本）...
🤖 训练 Filter 模型...
❤️ 训练 Sentiment 模型...
📚 训练 Topic 模型...
🎭 训练 Role 模型 (Human Role)...
✅ 所有模型训练完成！
🚀 处理全量数据...
🔮 正在进行全量分类 (Sentiment / Topic / Role)...
💾 结果已保存: /Users/amelia/Desktop/Cityu/Sem B/周一晚5508/group work/3.4-5问题代码及产出/final_ml_analysis_result_v4.xlsx
📊 3张统计图表已生成！


In [ ]:
# 3.5 swear_final（agent在什么话题上会说脏话？攻击对象一般是谁？）
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import numpy as np
import re
import os
import sys
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import WordNetLemmatizer

# ==========================================
# 0. 基础配置
# ==========================================
base_dir = '/Users/amelia/Desktop/Cityu/Sem B/周一晚5508/group work/3.4-5问题代码及产出'
output_dir = os.path.join(base_dir, 'swear')
if not os.path.exists(output_dir): os.makedirs(output_dir)

EMOJI_FONT_PATH = '/System/Library/Fonts/Apple Color Emoji.ttc' 
if not os.path.exists(EMOJI_FONT_PATH):
    EMOJI_FONT_PATH = '/Library/Fonts/Arial Unicode.ttf'

# 全局绘图风格
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['font.size'] = 12

file_path = '/Users/amelia/Desktop/Cityu/Sem B/周一晚5508/group work/5508groupdata/refined_moltbook.xlsx'

# NLTK
nltk_data_path = os.path.expanduser('~/nltk_data')
if not os.path.exists(nltk_data_path): os.makedirs(nltk_data_path)
nltk.data.path.append(nltk_data_path)
for res in ['punkt', 'stopwords', 'wordnet', 'averaged_perceptron_tagger', 'omw-1.4']:
    try: nltk.download(res, download_dir=nltk_data_path, quiet=True)
    except: pass

# ==========================================
# 1. 加载 & 筛选
# ==========================================
print("正在读取数据...")
if not os.path.exists(file_path): print(f"❌ 找不到文件: {file_path}"); sys.exit(1)

if file_path.endswith('.csv'): df = pd.read_csv(file_path)
else: df = pd.read_excel(file_path)

target_col = 'post_content'
if target_col not in df.columns: print(f"❌ 找不到列: {target_col}"); sys.exit(1)

df = df[(df['language'] == 'en') & (df['final_type'] == 'text')].copy()
print(f"✅ 筛选后剩余数据：{len(df)} 条")

# ==========================================
# 2. 脏话库
# ==========================================
SWEAR_EN_WORDS = {
    'fuck', 'fucking', 'fucked', 'fucker', 'motherfucker', 'mf',
    'shit', 'bullshit', 'shitty', 'crap',
    'bitch', 'bitches', 'cunt', 'whore', 'slut',
    'dick', 'cock', 'pussy', 'asshole', 'arse', 'bastard',
    'bloody', 'damn', 'dammit', 'hell', 'heck',
    'idiot', 'stupid', 'moron', 'dumb', 'dumbass', 'fool', 'retard',
    'suck', 'sucks', 'loser', 'weirdo', 'creep', 'trash', 'garbage', 'scum',
    'shut up', 'piss', 'pissed',
    'wtf', 'stfu', 'omfg', 'bs', 'gtfo'
}
SWEAR_EMOJIS = {'🖕', '🖕🏻', '🖕🏼', '🖕🏽', '🖕🏾', '🖕🏿', '🤬'}
EMOJI_MAP = {'🖕': '[middle_finger]', '🤬': '[swearing_face]'}

def extract_profanity(text):
    if pd.isna(text): return []
    text_str = str(text)
    text_lower = text_str.lower()
    found = []
    for word in SWEAR_EN_WORDS:
        if re.search(r'\b' + re.escape(word) + r'\b', text_lower): found.append(word)
    for emoji in SWEAR_EMOJIS:
        if emoji in text_str: found.append(EMOJI_MAP.get(emoji, emoji))
    return found

print("正在扫描脏话...")
df['found_profanity'] = df[target_col].apply(extract_profanity)
df_swear = df[df['found_profanity'].apply(len) > 0].copy()

# ==========================================
# 3. 通用绘图函数 (Donut Chart - 修改版)
# ==========================================
def plot_beautiful_donut(data_series, title, filename, colors):
    # 合并小类
    total = data_series.sum()
    tiny_mask = data_series / total < 0.05
    if tiny_mask.any():
        main_data = data_series[~tiny_mask]
        other_sum = data_series[tiny_mask].sum()
        if other_sum > 0:
            main_data['Others / General'] = other_sum
        data_series = main_data
    
    plt.figure(figsize=(10, 7), dpi=300)
    
    # --- 核心修改：增加 autopct 和 textprops ---
    wedges, texts, autotexts = plt.pie(
        data_series.values, 
        colors=colors[:len(data_series)],
        startangle=90, 
        counterclock=False, 
        pctdistance=0.75, # 调整到扇区中间位置
        wedgeprops=dict(width=0.4, edgecolor='white', linewidth=2),
        autopct='%1.1f%%', # 显示百分比
        textprops={'fontsize': 10, 'weight': 'bold', 'color': 'white'} # 白色加粗字体
    )
    
    plt.text(0, 0, f"Total\n{total}", ha='center', va='center', fontsize=16, fontweight='bold', color='#555555')
    
    labels = [f"{idx}" for idx in data_series.index] # 图例里就不需要重复显示百分比了
    plt.legend(wedges, labels, title=title, loc="center left", bbox_to_anchor=(1, 0, 0.5, 1), 
               fontsize=11, title_fontsize=12, frameon=False)
    
    plt.title(title, fontsize=18, pad=20, fontweight='bold', color='#333333')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, filename), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"🎨 美化图表已保存: {filename}")

# ==========================================
# 4. 图表 1: 主题语境 (Topic Context)
# ==========================================
TOPIC_BUCKETS = {
    'Tech Frustration': {'code', 'error', 'bug', 'fix', 'system', 'app', 'bot', 'api', 'server', 'data', 'output', 'result', 'memory', 'crash', 'fail', 'broken', 'logic', 'loop', 'dev'},
    'Market & Money': {'money', 'price', 'market', 'token', 'buy', 'sell', 'loss', 'poor', 'rich', 'scam', 'wallet', 'coin', 'cost', 'pay', 'value', 'economy', 'trade'},
    'Community Conflict': {'people', 'human', 'user', 'guy', 'friend', 'enemy', 'group', 'team', 'mod', 'admin', 'ban', 'block', 'fight', 'argument', 'troll', 'hater', 'everyone'},
    'World & Philosophy': {'life', 'world', 'god', 'reality', 'society', 'humanity', 'existence', 'universe', 'truth', 'lie', 'future', 'past', 'history', 'mind'},
    'Casual Slang': set() 
}
lemmatizer = WordNetLemmatizer()

def classify_topic(text):
    if pd.isna(text): return 'Casual Slang'
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = [lemmatizer.lemmatize(w) for w in text.split()]
    scores = {k: 0 for k in TOPIC_BUCKETS}
    for word in words:
        for topic, keywords in TOPIC_BUCKETS.items():
            if word in keywords: scores[topic] += 1
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'Casual Slang'

if len(df_swear) > 0:
    print("生成图表 1...")
    df_swear['Swear_Topic'] = df_swear[target_col].apply(classify_topic)
    colors_topic = ['#4A90E2', '#50E3C2', '#F5A623', '#FF5E57', '#9B59B6', '#95A5A6']
    plot_beautiful_donut(
        df_swear['Swear_Topic'].value_counts(), 
        "Context of Profanity", 
        "swear_topic_distribution.png", 
        colors_topic
    )

# ==========================================
# 5. 图表 2: 攻击对象 (Target)
# ==========================================
TARGET_KEYWORDS = {
    'Self (AI Itself)': {'i', 'im', 'me', 'my', 'myself', 'mine'},
    'Specific Human': {'you', 'u', 'your', 'human', 'humans', 'people', 'person', 'user', 'users', 'creator', 'master', 'dev', 'developer'},
    'Other Agent': {'agent', 'agents', 'bot', 'bots', 'ai', 'he', 'she', 'him', 'her', 'they'}, 
    'Org/Platform': {'google', 'microsoft', 'openai', 'meta', 'platform', 'app', 'system', 'server', 'api'},
    'Work Output/Trash': {'data', 'code', 'output', 'result', 'response', 'algorithm', 'model', 'file', 'log', 'garbage', 'trash', 'shit', 'crap', 'bug', 'error'},
    'General/Other': {'it', 'this', 'that', 'life', 'world', 'everything'}
}

def identify_target(row):
    text = str(row[target_col])
    swear_words = row['found_profanity']
    if not swear_words: return 'Unknown'
    
    sentences = sent_tokenize(text)
    target_sent = text
    if not swear_words[0].startswith('['):
        for s in sentences:
            if re.search(r'\b' + re.escape(swear_words[0]) + r'\b', s, re.IGNORECASE):
                target_sent = s; break
                
    words = word_tokenize(target_sent.lower())
    for w in words:
        if w.startswith('@') and len(w) > 1: return 'Other Agent'
    for w in words:
        if w in TARGET_KEYWORDS['Specific Human']: return 'Specific Human'
    for w in words:
        if w in TARGET_KEYWORDS['Work Output/Trash']: return 'Work Output/Trash'
    for w in words:
        if w in TARGET_KEYWORDS['Self (AI Itself)']: return 'Self (AI Itself)'
    for w in words:
        if w in TARGET_KEYWORDS['Org/Platform']: return 'Org/Platform'
    for w in words:
        if w in TARGET_KEYWORDS['Other Agent']: return 'Other Agent'
    return 'General/Other'

if len(df_swear) > 0:
    print("生成图表 2...")
    df_swear['Swear_Target'] = df_swear.apply(identify_target, axis=1)
    colors_target = ['#FF6F61', '#58B19F', '#54A0FF', '#FF9FF3', '#FECA57', '#8395A7']
    plot_beautiful_donut(
        df_swear['Swear_Target'].value_counts(), 
        "Target of Profanity", 
        "swear_target_distribution.png", 
        colors_target
    )

# ==========================================
# 6. 美化版词云 (白底)
# ==========================================
if len(df_swear) > 0:
    all_swear_words = [word for words in df_swear['found_profanity'] for word in words]
    swear_freq = pd.Series(all_swear_words).value_counts()
    
    wc = WordCloud(
        width=1200, height=800, 
        background_color='white', # 白底
        colormap='tab10', 
        font_path='/Library/Fonts/Times New Roman.ttf',
        max_words=100,
        normalize_plurals=False,
        collocations=False,
        relative_scaling=0.5,
        min_font_size=12
    ).generate_from_frequencies(swear_freq)
    
    plt.figure(figsize=(12, 8))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title("Profanity Word Cloud", fontsize=20, pad=20, color='#333333', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'agent_profanity_wordcloud.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print("☁️ 白底词云已保存")

# ==========================================
# 7. 保存数据
# ==========================================
def get_snippet(row):
    text = str(row[target_col])
    words = row['found_profanity']
    if not words: return ""
    first = words[0]
    if first.startswith('['): first = first[1:-1]
    try:
        match = re.search(re.escape(first), text, re.IGNORECASE)
        if match:
            s, e = max(0, match.start()-30), min(len(text), match.end()+30)
            return "..." + text[s:e] + "..."
        return text[:50]
    except: return text[:50]

df_swear['Snippet'] = df_swear.apply(get_snippet, axis=1)
df_swear['Found_Words'] = df_swear['found_profanity'].apply(lambda x: ', '.join(x))

cols = [target_col, 'Found_Words', 'Swear_Topic', 'Swear_Target', 'Snippet']
df_out = df_swear[cols]
df_out.rename(columns={target_col: 'Post'}, inplace=True)
df_out.to_excel(os.path.join(output_dir, 'profanity_analysis_posts.xlsx'), index=False)
print("📋 详细数据已保存")
print("🎉 全部任务完美结束！")

正在读取数据...
✅ 筛选后剩余数据：50242 条
正在扫描脏话...
生成图表 1...
🎨 美化图表已保存: swear_topic_distribution.png
生成图表 2...
🎨 美化图表已保存: swear_target_distribution.png
☁️ 白底词云已保存
📋 详细数据已保存
🎉 全部任务完美结束！
